In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')
from src.data_processor import DataProcessor
from src.popularity_predictor import PopularityPredictor

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("✓ Libraries loaded")

## 1. Load and Prepare Data

In [ ]:
# Load data
processor = DataProcessor()

try:
    df = pd.read_csv('../data/processed/music_data_clustered.csv')
    print(f"Loaded {len(df)} tracks")
except:
    df = processor.create_sample_data(n_samples=5000)
    print(f"Created {len(df)} synthetic tracks")

df.head()

In [ ]:
# Prepare features and target
feature_columns = ['danceability', 'energy', 'loudness', 'speechiness', 
                   'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
available_features = [f for f in feature_columns if f in df.columns]

print(f"Features: {available_features}")

X = df[available_features]
y = df['popularity']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=available_features)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Further split for validation (for ANN)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"\nTrain: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## 2. Train All Models

In [ ]:
# Initialize predictor
predictor = PopularityPredictor(random_state=42)
predictor.feature_names = available_features

# Train all models
print("Training models...\n")
predictor.train_all_models(
    X_train.values, y_train.values,
    X_val.values, y_val.values
)

## 3. Evaluate Models

In [ ]:
# Evaluate on test set
results = predictor.evaluate(X_test.values, y_test.values)

# Display comparison
comparison_df = predictor.compare_models()
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))
print(f"\n✓ Best Model: {predictor.best_model_name}")

In [ ]:
# Visual comparison
fig = predictor.plot_comparison()
plt.savefig('../data/processed/model_comparison.png', dpi=150)
plt.show()

## 4. Detailed Analysis - Best Model

In [ ]:
# Predictions vs Actual
best_model = predictor.best_model_name
fig = predictor.plot_predictions(y_test.values, best_model)
plt.savefig('../data/processed/predictions_scatter.png', dpi=150)
plt.show()

In [ ]:
# Residual analysis
y_pred = results[best_model]['predictions']
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residual distribution
axes[0].hist(residuals, bins=50, edgecolor='white')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution')
axes[0].axvline(x=0, color='r', linestyle='--')

# Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

print(f"Residual Stats:")
print(f"  Mean: {residuals.mean():.4f}")
print(f"  Std: {residuals.std():.4f}")

## 5. Feature Importance

In [ ]:
# Feature importance from Random Forest
if 'random_forest' in predictor.models:
    importance_df = predictor.get_feature_importance('random_forest')
    
    print("Feature Importance (Random Forest):")
    print(importance_df.to_string(index=False))
    
    fig = predictor.plot_feature_importance('random_forest')
    plt.savefig('../data/processed/feature_importance.png', dpi=150)
    plt.show()

In [ ]:
# Compare with Gradient Boosting
if 'gradient_boosting' in predictor.models:
    importance_gb = predictor.get_feature_importance('gradient_boosting')
    
    # Merge importances
    importance_df = importance_df.rename(columns={'importance': 'RF_importance'})
    importance_df = importance_df.merge(
        importance_gb.rename(columns={'importance': 'GB_importance'}),
        on='feature'
    )
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(importance_df))
    width = 0.35
    
    ax.barh(x - width/2, importance_df['RF_importance'], width, label='Random Forest')
    ax.barh(x + width/2, importance_df['GB_importance'], width, label='Gradient Boosting')
    
    ax.set_yticks(x)
    ax.set_yticklabels(importance_df['feature'])
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance Comparison')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Cross-Validation

In [ ]:
# Cross-validation for best models
cv_results = {}

# Combine train and val for CV
X_cv = np.vstack([X_train.values, X_val.values])
y_cv = np.hstack([y_train.values, y_val.values])

for model_name in ['random_forest', 'gradient_boosting', 'linear_regression']:
    if model_name in predictor.models:
        cv_result = predictor.cross_validate(X_cv, y_cv, model_name, cv=5)
        cv_results[model_name] = cv_result
        print(f"{model_name}: R² = {cv_result['mean_r2']:.4f} (+/- {cv_result['std_r2']:.4f})")

In [ ]:
# CV results visualization
if cv_results:
    cv_df = pd.DataFrame({
        'Model': list(cv_results.keys()),
        'Mean R²': [r['mean_r2'] for r in cv_results.values()],
        'Std R²': [r['std_r2'] for r in cv_results.values()]
    })
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(cv_df['Model'], cv_df['Mean R²'], yerr=cv_df['Std R²'], capsize=5)
    ax.set_ylabel('R² Score')
    ax.set_title('5-Fold Cross-Validation Results')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. ANN Training History

In [ ]:
# Plot ANN training history
if 'ann_history' in predictor.models:
    history = predictor.models['ann_history']
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (MSE)')
    axes[0].set_title('ANN Training Loss')
    axes[0].legend()
    
    # MAE
    axes[1].plot(history.history['mae'], label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Val MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('ANN Training MAE')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('../data/processed/ann_training.png', dpi=150)
    plt.show()

## 8. Save Best Model

In [ ]:
# Save models
import os

# Save Random Forest (typically best for this task)
if 'random_forest' in predictor.models:
    predictor.save_model('random_forest', '../models/popularity/random_forest.pkl')

# Save ANN
if 'ann' in predictor.models:
    predictor.save_model('ann', '../models/popularity/ann_model.keras')

print("\n✓ Models saved successfully")

## Summary

### Model Performance:
| Model | R² | RMSE | MAE |
|-------|----|----- |-----|
| Random Forest | Best for non-linear | Moderate | Good |
| Gradient Boosting | Often second best | Similar | Similar |
| ANN | Variable | Depends on tuning | - |
| Linear Regression | Baseline | Higher | Higher |

### Key Insights:
1. **Best Features**: Energy, loudness, and danceability are typically most predictive
2. **Model Choice**: Random Forest provides good balance of accuracy and interpretability
3. **Limitations**: Popularity is inherently hard to predict from audio features alone